# Milestone 6: From one agent to a team of agents
**Project: Northwind Knowledge Assistant** | **Presenter: Nagarjuna Naidu** | Day 5, before the case study (about 15 minutes)

Goal: see the patterns the case study asks you to design, working in about 80 lines of plain Python: a **supervisor** that routes work to **specialist agents**, **shared state** (short-term memory), a **long-term memory** with human validation, a **human approval gate**, **failure handling**, and a **trace** you can read.

The example here is a travel-policy exception request, not the case study. At the end you fill in a design sheet for the **IT operations platform** yourself.

In [ ]:
import time, json, re
from nka.tools import search_docs, query_sql

print("Imports ready")

## 1. Shared state: the short-term memory of one request
Every agent reads and writes the same state object. It lives for one request and is cleared when the request closes.

In [ ]:
def new_state(request):
    return {"request": request, "facts": {}, "notes": [], "confidence": 1.0,
            "risk": "low", "decision": None, "trace": [], "calls": 0}

def log(state, agent, event, detail=""):
    state["trace"].append({"t": round(time.time() % 1000, 2), "agent": agent, "event": event, "detail": str(detail)[:90]})

## 2. Specialist agents
Each agent does one job and uses its own tools. Small, focused agents are easier to test, to restrict and to replace.

In [ ]:
FAIL_ONCE = {"data": False}        # used later to simulate a failing tool

def data_agent(state):
    """Looks up the booking facts (tool: read-only SQL)."""
    if FAIL_ONCE["data"]:
        FAIL_ONCE["data"] = False
        raise TimeoutError("database timeout")
    r = state["request"]
    rows = json.loads(query_sql(f"SELECT traveller, nights, price_per_night FROM bookings WHERE city='{r['city']}' AND traveller='{r['traveller']}'"))
    state["facts"]["booking"] = rows[0] if rows else None
    state["confidence"] *= 1.0 if rows else 0.4
    return "booking found" if rows else "no booking found"

def policy_agent(state):
    """Retrieves the policy text (tool: document search)."""
    text = search_docs(f"{state['request']['city']} hotel cap per night", k=1)
    m = re.search(r"GBP\s*(\d+)", text)
    state["facts"]["cap"] = int(m.group(1)) if m else None
    state["confidence"] *= 0.95 if m else 0.5
    return f"cap = {state['facts']['cap']}"

def risk_agent(state):
    """Compliance: classifies the action into a risk tier. No LLM needed, rules are auditable."""
    b, cap = state["facts"].get("booking"), state["facts"].get("cap")
    if not b or cap is None:
        state["risk"] = "unknown"; state["facts"]["overage"] = None; return "cannot classify"
    over = max(0, (b["price_per_night"] - cap) * b["nights"])
    state["facts"]["overage"] = over
    state["risk"] = "low" if over == 0 else "medium" if over <= 100 else "high"
    return f"overage {over} -> risk {state['risk']}"

AGENTS = {"data": data_agent, "policy": policy_agent, "risk": risk_agent}

## 3. The supervisor
The supervisor does not do the work. It decides **which agent runs next**, enforces limits, and decides when to stop or hand over to a human.

This one is a simple rule table. Many real systems let an LLM choose the next agent; the control flow around it stays the same.

In [ ]:
def next_agent(state):
    f = state["facts"]
    if "booking" not in f: return "data"
    if "cap" not in f: return "policy"
    if "overage" not in f: return "risk"
    return None

def run_agent_safely(name, state, retries=1):
    """Failure handling: retry once, then record the failure and carry on without that agent's result."""
    for attempt in range(retries + 1):
        try:
            state["calls"] += 1
            out = AGENTS[name](state)
            log(state, name, "ok", out); return True
        except Exception as e:
            log(state, name, "error", f"{type(e).__name__}: {e} (attempt {attempt + 1})")
    state["confidence"] *= 0.3
    state["facts"].setdefault({"data": "booking", "policy": "cap", "risk": "overage"}[name], None)
    return False

def supervise(request, max_steps=6, min_confidence=0.6, approve=lambda s: False):
    state = new_state(request)
    for _ in range(max_steps):                                  # step limit
        name = next_agent(state)
        if name is None: break
        run_agent_safely(name, state)
    else:
        log(state, "supervisor", "stop", "step limit reached")
    # decision, with the human-in-the-loop boundary
    if state["confidence"] < min_confidence or state["risk"] == "unknown":
        state["decision"] = "ESCALATE to a human (low confidence)"
    elif state["risk"] == "low":
        state["decision"] = "AUTO-APPROVE"
    elif state["risk"] == "medium":
        state["decision"] = "RECOMMEND approval (a human confirms)"
    else:
        state["decision"] = "APPROVED by human" if approve(state) else "HELD for human approval"
    log(state, "supervisor", "decision", state["decision"])
    return state

def show(state):
    print(f"{state['request']}\n  decision  : {state['decision']}\n  risk/conf : {state['risk']} / {state['confidence']:.2f}   agent calls: {state['calls']}")
    for e in state["trace"]: print(f"    {e['agent']:<11}{e['event']:<9}{e['detail']}")
    print()

## 4. Three runs
1. A normal request, 2. a request that needs a human, 3. a request where a tool fails once (retry), then a case where it fails twice (escalate).

In [ ]:
show(supervise({"city": "London", "traveller": "A. Shah"}))      # inside the cap
show(supervise({"city": "London", "traveller": "D. Rossi"}))     # over the cap: high risk, held for a human
FAIL_ONCE["data"] = True
show(supervise({"city": "London", "traveller": "B. Okafor"}))    # tool fails once, retry succeeds

**Look at the trace.** Each line is one agent step. This is what you would send to a logging or tracing tool in production. Counting `agent calls` is the simplest cost measure.

In [ ]:
def always_fail(state): raise TimeoutError("database timeout")
AGENTS["data"] = always_fail                                     # now the tool is down
show(supervise({"city": "London", "traveller": "C. Meyer"}))     # escalated, not guessed
AGENTS["data"] = data_agent                                      # put it back

## 5. Long-term memory with human validation
Short-term memory (the state) disappears after each request. Long-term memory keeps **resolved cases** so that next time the system can retrieve similar ones. Rule: **only a human-validated outcome is written**.

In [ ]:
CASE_MEMORY = []                      # in a real system: a vector database with metadata

def remember(state, validated_by):
    if not validated_by: print("not stored: nobody validated this outcome"); return
    CASE_MEMORY.append({"request": state["request"], "decision": state["decision"], "validated_by": validated_by})
    print("stored case", len(CASE_MEMORY))

def similar_cases(request):
    return [c for c in CASE_MEMORY if c["request"]["city"] == request["city"]]   # real systems use embeddings

s = supervise({"city": "London", "traveller": "D. Rossi"}, approve=lambda s: True)
remember(s, validated_by=None)        # refused
remember(s, validated_by="finance manager")
print("similar cases for London:", [c["decision"] for c in similar_cases({"city": "London"})])

## 6. How this maps to the frameworks you will hear about
| In this notebook | LangGraph | Other names |
|---|---|---|
| `state` dictionary | the graph state | shared context |
| each `*_agent` function | a node | worker, specialist |
| `supervise` and `next_agent` | the graph and its conditional edges | supervisor, orchestrator |
| `approve` gate | an interrupt / human-in-the-loop step | approval tier |
| `run_agent_safely` | retry policy on a node | failure handling |
| `trace` | tracing (for example LangSmith, OpenTelemetry) | observability |
| `CASE_MEMORY` | a store behind the graph | long-term memory |

The frameworks add persistence, streaming and tooling. The **ideas** are the ones you just ran.

## 7. YOUR TASK: design sheet for the IT operations platform
Do not write code. Fill in the sheet below for **this afternoon's case study**. Use short phrases. Then redraw it as your architecture diagram.

Think about the payment incident (high latency, more errors, database timeouts): who does what, in which order, and who is allowed to do what.

In [ ]:
design = {
    "agents (name -> its one job)":              {},     # TODO: for example {"agent name": "one sentence job"}
    "tools each agent may use":                  {},     # TODO: agent -> list of tools (think MCP servers)
    "who decides the next step (supervisor)":    "",     # TODO
    "short-term memory holds":                   "",     # TODO
    "long-term memory holds, and who validates": "",     # TODO
    "risk tiers and who approves each":          {},     # TODO: low / medium / high -> what happens
    "failure handling (tool down, low confidence)": "",  # TODO
    "what you trace, and one cost control":      "",     # TODO
}

def check(sheet):
    missing = [k for k, v in sheet.items() if not v]
    print("Sheet complete." if not missing else "Still to fill in:\n  - " + "\n  - ".join(missing))

check(design)